# Load cleaned issues data

In [10]:
import os
import pandas as pd

# Define input directory
input_dir = "issues-cleanedData"

# Initialize dictionary to hold DataFrames
issues_cleaned_data = {}

# Process each CSV file
for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        file_path = os.path.join(input_dir, filename)
        df = pd.read_csv(file_path)

        # Check if required columns are present
        if all(col in df.columns for col in ['issue_state', 'days_elapsed', 'status']):
            # Filter out rows where 'issue_state' is 'NA'
            df = df.dropna(subset=['issue_state'])
            
            # Keep only the required columns
            df = df[['issue_state', 'days_elapsed', 'status']]
            
            # Add DataFrame to dictionary using filename (without .csv) as key
            issues_cleaned_data[filename[:-11]] = df


issues_cleaned_data['age']


,issue_state,days_elapsed,status
0,CLOSED,4,Graduated
3,CLOSED,21,Graduated
9,CLOSED,1,Graduated
12,CLOSED,7,Graduated
14,CLOSED,287,Graduated
...,...,...,...
4945,CLOSED,9,Graduated
4946,OPEN,24,Graduated
4952,OPEN,12,Graduated
4954,OPEN,12,Graduated


# Import scraper data

In [11]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


def clean_csv_files(folder_path):
  cleaned_dataframes = {}

  # List of columns to drop
  columns_to_drop = [
      "status", "start_date", "end_date", "window_start_date", "window_end_date",
      "emails", "devs", "emails_thread_starter", "emails_thread_starter_word_count",
      "emails_thread_starter_characters", "emails_threads", "emails_threads_word_count",
      "emails_threads_characters", "emails_no_replies", "emails_no_replies_word_count",
      "emails_no_replies_characters", "emails_jira", "most_complex_unit_loc",
      "most_complex_unit_mcabe_index", "total_number_of_files", "number_of_files_main",
      "lines_of_code_main", "number_of_files_test", "lines_of_code_test",
      "test_vs_main_lines_of_code_percentage", "number_of_files_generated",
      "lines_of_code_generated", "number_of_files_build_and_deployment",
      "lines_of_code_build_and_deployment", "negligible_risk_file_size_count",
      "low_risk_file_size_count", "medium_risk_file_size_count", "high_risk_file_size_count",
      "very_high_risk_file_size_count", "negligible_risk_file_size_loc", "low_risk_file_size_loc",
      "medium_risk_file_size_loc", "high_risk_file_size_loc", "very_high_risk_file_size_loc",
      "number_of_units", "lines_of_code_in_units", "lines_of_code_outside_units",
      "unit_size_negligible_risk_loc", "unit_size_negligible_risk_count", "unit_size_low_risk_loc",
      "unit_size_low_risk_count", "unit_size_medium_risk_loc", "unit_size_medium_risk_count",
      "unit_size_high_risk_loc", "unit_size_high_risk_count", "unit_size_very_high_risk_loc",
      "unit_size_very_high_risk_count", "conditional_complexity_negligible_risk_loc",
      "conditional_complexity_negligible_risk_count", "conditional_complexity_low_risk_loc",
      "conditional_complexity_low_risk_count", "conditional_complexity_medium_risk_loc",
      "conditional_complexity_medium_risk_count", "conditional_complexity_high_risk_loc",
      "conditional_complexity_high_risk_count", "conditional_complexity_very_high_risk_loc",
      "conditional_complexity_very_high_risk_count", "conditional_complexity_high_plus_risk_count",
      "conditional_complexity_high_plus_risk_loc", "number_of_contributors",
      "duplication_number_of_duplicates", "duplication_number_of_files_with_duplicates",
      "duplication_number_of_duplicated_lines", "duplication_percentage", "unit_duplicates_count", "releases"
  ]

  for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
      file_path = os.path.join(folder_path, filename)

      # Load CSV file
      df = pd.read_csv(file_path)

      # Drop specified columns
      df = df.drop(
          columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

      key = os.path.splitext(filename)[0]
      cleaned_dataframes[key] = df

  return cleaned_dataframes

folder_path = "scraper-output"
cleaned_data = clean_csv_files(folder_path)

for key, df in cleaned_data.items():
    # Replace NaN values in numerical columns with 0
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(0)

    # Replace NaN and blank/empty values in 'programming_lang' column with the mode
    if 'programming_lang' in df.columns:
        # Calculate mode value
        mode_value = df['programming_lang'].mode()[0] if not df['programming_lang'].mode().empty else 'Unknown'
        
        # Replace NaN values with the mode
        df['programming_lang'] = df['programming_lang'].fillna(mode_value)
        
        # Replace blank or whitespace-only values with the mode
        df['programming_lang'] = df['programming_lang'].replace(r'^\s*$', mode_value, regex=True)

status_data = pd.read_csv("project-status.csv")

# Filter out projects with fewer than 10 data points
cleaned_data = {project: df for project,
                df in cleaned_data.items() if len(df) >= 10}

def merge_status(cleaned_data, status_data):
  status_dict = status_data.set_index('project')['status'].to_dict()
  for project, df in cleaned_data.items():
    df['status'] = status_dict.get(project, 'Unknown')
  return cleaned_data

cleaned_data = merge_status(cleaned_data, status_data)

for key, df in cleaned_data.items():
  cleaned_data[key] = df[["commits", "authors",
                                      "committers", "status"]]


# Keep scraper output only for projects for which we have issue data

In [12]:
# Get the keys from issues_cleaned_data
issues_cleaned_data_keys = set(issues_cleaned_data.keys())

# Filter cleaned_data to keep only the keys present in issues_cleaned_data
cleaned_data = {key: value for key, value in cleaned_data.items() if key in issues_cleaned_data_keys}

if 'kiso' in issues_cleaned_data.keys():
    del issues_cleaned_data['kiso'] 

# Deepcopy cleaned_data to create cleaned_data_transformer
import copy
cleaned_data_transformer = copy.deepcopy(cleaned_data)

# Print the filtered cleaned_data dictionary
print("Filtered cleaned_data:")
print(len(cleaned_data_transformer), len(issues_cleaned_data))


Filtered cleaned_data:
87 87


In [13]:
issues_cleaned_data['age']

,issue_state,days_elapsed,status
0,CLOSED,4,Graduated
3,CLOSED,21,Graduated
9,CLOSED,1,Graduated
12,CLOSED,7,Graduated
14,CLOSED,287,Graduated
...,...,...,...
4945,CLOSED,9,Graduated
4946,OPEN,24,Graduated
4952,OPEN,12,Graduated
4954,OPEN,12,Graduated


# LSTM model on combined issues and scraper data

In [14]:
import pandas as pd
import numpy as np
import tensorflow as tf
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Flatten, LSTM, Dense, Masking, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set device preference (CUDA > MPS > CPU)
if tf.config.list_physical_devices('GPU'):
    device_name = 'GPU'
elif tf.config.list_physical_devices('MPS'):
    device_name = 'MPS'
else:
    device_name = 'CPU'

print(f"Using {device_name} for training.")

# Encode 'OPEN' as 1 and 'CLOSED' as 0 for issue_state in issues_cleaned_data
for project in issues_cleaned_data:
    issues_cleaned_data[project]['issue_state'] = issues_cleaned_data[project]['issue_state'].map({
                                                                                                    'OPEN': 1, 'CLOSED': 0})

    # Standard scale 'days_elapsed'
    scaler = StandardScaler()
    issues_cleaned_data[project]['days_elapsed'] = scaler.fit_transform(
        issues_cleaned_data[project]['days_elapsed'].values.reshape(-1, 1))

# Standard scale all features except 'status' for cleaned_data_transformer
for project in cleaned_data_transformer:
    features = cleaned_data_transformer[project].drop(
        columns=['status'])  # Exclude 'status'
    scaler = StandardScaler()
    cleaned_data_transformer[project][features.columns] = scaler.fit_transform(
        features)

# Prepare separate inputs for each dictionary
X1, X2, y = [], [], []
for project in cleaned_data_transformer:
    df1 = cleaned_data_transformer[project].drop(columns=['status'])
    df2 = issues_cleaned_data[project].drop(columns=['status'])
    X1.append(df1.values)
    X2.append(df2.values)
    y.append(cleaned_data_transformer[project]['status'].iloc[0])

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Pad sequences independently for both inputs
X1_padded = pad_sequences(X1, dtype='float32', padding='post')
X2_padded = pad_sequences(X2, dtype='float32', padding='post')

# Train-test-validation split (70/20/10)
(X1_train, X1_temp, X2_train, X2_temp, y_train, y_temp) = train_test_split(
    X1_padded, X2_padded, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42
)
(X1_val, X1_test, X2_val, X2_test, y_val, y_test) = train_test_split(
    X1_temp, X2_temp, y_temp, test_size=0.33, stratify=y_temp, random_state=42
)

# Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# Define objective function for Optuna
def objective(trial):
    # Hyperparameters to tune
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    dropout_rate = trial.suggest_uniform('dropout_rate', 0.1, 0.5)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)

    # Define model
    input1 = Input(shape=(X1_padded.shape[1], X1_padded.shape[2]))
    input2 = Input(shape=(X2_padded.shape[1], X2_padded.shape[2]))

    masked1 = Masking(mask_value=0.0)(input1)
    masked2 = Masking(mask_value=0.0)(input2)

    lstm1 = LSTM(32, dropout=dropout_rate)(masked1)
    lstm2 = LSTM(32, dropout=dropout_rate)(masked2)

    concat = Concatenate()([lstm1, lstm2])
    flatten = Flatten()(concat)  # Flattening before passing to Dense
    dense = Dense(16, activation='relu')(flatten)
    dropout = Dropout(dropout_rate)(dense)
    output = Dense(1, activation='sigmoid')(dropout)

    model = Model(inputs=[input1, input2], outputs=output)
    optimizer = Adam(learning_rate=learning_rate, decay=weight_decay)
    model.compile(optimizer=optimizer, loss='binary_crossentropy',
                  metrics=['accuracy'])

    # Use the selected device
    with tf.device(f'/device:{device_name}:0'):
        # Train the model and print loss/acc for every epoch
        for epoch in range(15):
            history = model.fit(
                [X1_train, X2_train], y_train,
                validation_data=([X1_val, X2_val], y_val),
                epochs=1,
                batch_size=64,
                class_weight=class_weight_dict,
                verbose=0
            )
            train_loss = history.history['loss'][0]
            train_acc = history.history['accuracy'][0]
            val_loss = history.history['val_loss'][0]
            val_acc = history.history['val_accuracy'][0]
            print(f"Epoch {epoch+1:02d} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        # Evaluate on test data and print test accuracy
        test_loss, test_acc = model.evaluate([X1_test, X2_test], y_test, verbose=0)
        print(
            f"Test Accuracy: {test_acc:.4f} | Hyperparams: LR={learning_rate:.5f}, Dropout={dropout_rate:.2f}, WD={weight_decay:.6f}")

        # Confusion Matrix for each hyperparam combination
        y_pred = (model.predict([X1_test, X2_test]) > 0.5).astype("int32")
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm, display_labels=label_encoder.classes_)
        disp.plot(cmap='Blues')
        plt.title(f'Confusion Matrix (Test Acc: {test_acc:.4f})')
        plt.show()

    return 1 - test_acc  # Minimize error


# Run Optuna optimization
study = optuna.create_study(
    direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)

# Best hyperparameters
best_params = study.best_params
print(f"\nBest Hyperparameters: {best_params}")

# Evaluate best model on test data
best_test_acc = 1 - study.best_value
print(f"Best Test Accuracy: {best_test_acc:.4f}")


ModuleNotFoundError: No module named 'tensorflow'